# 第3章 马尔可夫决策过程与贝尔曼方程

本章介绍马尔可夫决策过程（MDP）的核心概念、贝尔曼方程与最优策略。

## 1. 环境配置

In [ ]:
import numpy as np
import random
import matplotlib.pyplot as plt
plt.rcParams['font.sans-serif'] = ['SimHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False
print("环境配置完成")

## 2. 编程题1: 自定义MDP环境与价值迭代

### 题目描述
实现一个自定义的MDP环境（如简化版Gridworld），包括状态空间、动作空间、转移概率和奖励函数，并实现价值迭代算法求解最优策略。

In [ ]:
class GridworldMDP:
    """简化版Gridworld MDP环境"""
    def __init__(self, n=3):
        self.n = n
        self.states = [(i, j) for i in range(n) for j in range(n)]
        self.terminal_state = (n-1, n-1)
        self.actions = ['up', 'down', 'left', 'right']
        
    def step(self, state, action):
        """状态转移"""
        x, y = state
        if action == 'up' and x > 0:
            x -= 1
        elif action == 'down' and x < self.n - 1:
            x += 1
        elif action == 'left' and y > 0:
            y -= 1
        elif action == 'right' and y < self.n - 1:
            y += 1
        next_state = (x, y)
        reward = 1 if next_state == self.terminal_state else -0.1
        done = next_state == self.terminal_state
        return next_state, reward, done

def value_iteration(env, gamma=0.9, theta=1e-6, max_iter=1000):
    """价值迭代算法"""
    V = {s: 0 for s in env.states}
    
    for _ in range(max_iter):
        delta = 0
        new_V = V.copy()
        
        for s in env.states:
            if s == env.terminal_state:
                continue
            max_value = -np.inf
            for a in env.actions:
                next_s, r, done = env.step(s, a)
                q_value = r + gamma * V[next_s]
                max_value = max(max_value, q_value)
            new_V[s] = max_value
            delta = max(delta, abs(new_V[s] - V[s]))
        
        V = new_V
        if delta < theta:
            break
    
    policy = {}
    for s in env.states:
        if s == env.terminal_state:
            continue
        best_action = env.actions[0]
        best_value = -np.inf
        for a in env.actions:
            next_s, r, _ = env.step(s, a)
            q_value = r + gamma * V[next_s]
            if q_value > best_value:
                best_value = q_value
                best_action = a
        policy[s] = best_action
    
    return V, policy

env = GridworldMDP(n=3)
V, policy = value_iteration(env, gamma=0.9)

print("=== Gridworld MDP价值迭代结果 ===")
print(f"状态数: {len(env.states)}")
print(f"最优策略: {policy}")

V_grid = np.array([V[s] for s in env.states]).reshape(3, 3)
print("价值函数:\n", V_grid)

## 3. 编程题2: 策略迭代算法

### 题目描述
编程实现策略迭代算法，包括策略评估和策略改进两个步骤，并在自定义MDP上测试算法收敛性。

In [ ]:
def policy_evaluation(env, policy, gamma=0.9, theta=1e-6, max_iter=1000):
    """策略评估"""
    V = {s: 0 for s in env.states}
    
    for _ in range(max_iter):
        delta = 0
        new_V = V.copy()
        
        for s in env.states:
            if s == env.terminal_state:
                continue
            a = policy[s]
            next_s, r, done = env.step(s, a)
            new_V[s] = r + gamma * V[next_s]
            delta = max(delta, abs(new_V[s] - V[s]))
        
        V = new_V
        if delta < theta:
            break
    
    return V

def policy_improvement(env, V, gamma=0.9):
    """策略改进"""
    policy = {}
    stable = True
    
    for s in env.states:
        if s == env.terminal_state:
            continue
        old_action = policy.get(s, None)
        best_action = env.actions[0]
        best_value = -np.inf
        
        for a in env.actions:
            next_s, r, _ = env.step(s, a)
            q_value = r + gamma * V[next_s]
            if q_value > best_value:
                best_value = q_value
                best_action = a
        policy[s] = best_action
        
        if old_action and old_action != best_action:
            stable = False
    
    return policy, stable

def policy_iteration(env, gamma=0.9, max_iter=100):
    """策略迭代算法"""
    policy = {s: np.random.choice(env.actions) for s in env.states}
    policy.pop(env.terminal_state, None)
    
    for i in range(max_iter):
        V = policy_evaluation(env, policy, gamma)
        policy, stable = policy_improvement(env, V, gamma)
        
        if stable:
            print(f"策略在第{i+1}次迭代后收敛")
            break
    
    return V, policy

env = GridworldMDP(n=3)
V, policy = policy_iteration(env, gamma=0.9)

print("=== 策略迭代结果 ===")
print(f"最优策略: {policy}")
V_grid = np.array([V[s] for s in env.states]).reshape(3, 3)
print("价值函数:\n", V_grid)

## 4. 编程题3: NumPy实现价值迭代与可视化

### 题目描述
使用NumPy实现价值迭代算法，编写函数展示价值函数，编写函数展示最优策略。

In [ ]:
def value_iteration_numpy(n=3, gamma=0.9, theta=1e-6):
    """NumPy实现价值迭代"""
    n_states = n * n
    V = np.zeros(n_states)
    
    P = np.zeros((n_states, 4))
    R = np.zeros((n_states, 4))
    
    for i in range(n):
        for j in range(n):
            s = i * n + j
            if i == n-1 and j == n-1:
                continue
            for a_idx, a in enumerate(['up', 'down', 'left', 'right']):
                ni, nj = i, j
                if a == 'up' and i > 0: ni = i - 1
                elif a == 'down' and i < n-1: ni = i + 1
                elif a == 'left' and j > 0: nj = j - 1
                elif a == 'right' and j < n-1: nj = j + 1
                next_s = ni * n + nj
                P[s, a_idx] = next_s
                R[s, a_idx] = 1 if ni == n-1 and nj == n-1 else -0.1
    
    for _ in range(1000):
        V_new = np.max(R + gamma * V[P.astype(int)], axis=1)
        if np.max(np.abs(V_new - V)) < theta:
            break
        V = V_new
    
    policy_idx = np.argmax(R + gamma * V[P.astype(int)], axis=1)
    policy = [['up', 'down', 'left', 'right'][p] for p in policy_idx]
    
    return V, policy

V, policy = value_iteration_numpy(n=3, gamma=0.9)

V_grid = V.reshape(3, 3)
print("价值函数:\n", V_grid)
print("最优策略:", policy)

In [ ]:
def visualize_value_function(V, n=3):
    """可视化价值函数"""
    V_grid = V.reshape(n, n)
    fig, ax = plt.subplots(figsize=(6, 6))
    im = ax.imshow(V_grid, cmap='YlOrRd', origin='lower')
    for i in range(n):
        for j in range(n):
            ax.text(j, i, f'{V_grid[i,j]:.2f}', ha='center', va='center', fontsize=10)
    ax.set_title('状态价值函数')
    ax.set_xlabel('列')
    ax.set_ylabel('行')
    plt.colorbar(im, ax=ax)
    plt.show()

def visualize_policy(policy, n=3):
    """可视化策略"""
    arrow_map = {'up': '↑', 'down': '↓', 'left': '←', 'right': '→'}
    fig, ax = plt.subplots(figsize=(6, 6))
    ax.set_xlim(-0.5, n-0.5)
    ax.set_ylim(-0.5, n-0.5)
    ax.set_xticks(range(n))
    ax.set_yticks(range(n))

    for idx, p in enumerate(policy):
        ax.text(idx % n, idx // n, arrow_map[p], ha='center', va='center', fontsize=20)
    ax.set_title('最优策略 (箭头方向)')
    ax.grid(True)
    plt.show()

visualize_value_function(V, n=3)

## 5. 编程题4: 不同折扣因子对最优策略的影响

### 题目描述
修改价值迭代代码，观察不同折扣因子γ对最优策略的影响，分析γ的取值与智能体"短视"程度的关系。

In [ ]:
gamma_values = [0.1, 0.5, 0.9, 0.99]
results = {}

for gamma in gamma_values:
    V, policy = value_iteration_numpy(n=3, gamma=gamma)
    results[gamma] = {'V': V, 'policy': policy}
    print(f"gamma = {gamma}: 策略 = {policy}")

fig, axes = plt.subplots(2, 2, figsize=(10, 10))
for idx, gamma in enumerate(gamma_values):
    i, j = idx // 2, idx % 2
    V_grid = results[gamma]['V'].reshape(3, 3)
    im = axes[i, j].imshow(V_grid, cmap='YlOrRd', origin='lower')
    axes[i, j].set_title(f'γ = {gamma}')
    plt.colorbar(im, ax=axes[i, j])

plt.suptitle('不同折扣因子γ的价值函数', fontsize=14)
plt.tight_layout()
plt.show()

print("结论: γ越小智能体越短视，γ越大智能体越重视长期收益")

## 6. 本章小结

本章介绍了马尔可夫决策过程与贝尔曼方程:
1. MDP: 五元组⟨S, A, P, R, γ⟩
2. 价值函数: 状态价值函数V(s)和动作价值函数Q(s,a)
3. 贝尔曼方程: 价值函数的递归定义
4. 价值迭代与策略迭代算法
5. 最优策略与折扣因子的影响